# Notebook 05 — Gradient Boosting Suite

**Goal:** Train LightGBM, XGBoost, and CatBoost with GroupKFold CV. Tune LightGBM with Optuna. Save per-fold model artifacts and OOF predictions for ensemble (Notebook 09).

**Inputs:** `data/processed/train_features.parquet`, `data/processed/fold_assignments.parquet`  
**Outputs:** `data/processed/oof_predictions.parquet`, `models/lgbm_fold{0..4}.pkl`, `models/xgb_fold{0..4}.pkl`, `models/catboost_fold{0..4}.pkl`

## Models
| Model | Key design choice |
|-------|------------------|
| LightGBM | Leaf-wise growth; Optuna-tuned (100 trials); primary model |
| XGBoost | Level-wise growth; `tree_method='hist'`; validates LightGBM |
| CatBoost | Ordered target encoding built-in; passes Race, Driver, Compound as raw categoricals |

## Feature sets
- **LightGBM / XGBoost:** 26 base features + 3 fold-safe target encodings (Race, Driver, Driver avg stint length) = 29 features  
- **CatBoost:** 26 base features + Race, Driver, Compound as native cat_features = 29 features (no manual encoding needed)

In [1]:
from pathlib import Path
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import optuna
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score

optuna.logging.set_verbosity(optuna.logging.ERROR)
warnings.filterwarnings('ignore')

cwd = Path.cwd()
if cwd.name == 'notebooks' or 'notebooks' in str(cwd):
    while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
        cwd = cwd.parent
project_root  = cwd
processed_dir = project_root / 'data' / 'processed'
models_dir    = project_root / 'models'
models_dir.mkdir(exist_ok=True)

assert (processed_dir / 'train_features.parquet').exists(),   'Run Notebook 02 first'
assert (processed_dir / 'fold_assignments.parquet').exists(), 'Run Notebook 03 first'
print(f'Project root: {project_root}')

c:\Repos\predict-f1-pit-stops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: c:\Repos\predict-f1-pit-stops


In [2]:
df    = pd.read_parquet(processed_dir / 'train_features.parquet')
folds = pd.read_parquet(processed_dir / 'fold_assignments.parquet')
df    = df.merge(folds, on='id', how='left')
df    = df.sort_values(['Race', 'Year', 'LapNumber']).reset_index(drop=True)

BASE_FEATURES = [
    'TyreLife_normalized_by_compound', 'TyreLife_sq',
    'is_wet_tyre', 'compound_ordinal',
    'laps_remaining', 'is_testing_session',
    'Stint', 'Position',
    'Cumulative_Degradation_winsorized', 'Degradation_rate', 'Degradation_acceleration',
    'LapTime_lag1', 'LapTime_lag2', 'LapTime_lag3',
    'LapTime_Delta_lag1', 'LapTime_Delta_lag2', 'LapTime_Delta_lag3',
    'LapTime_rolling_mean_3', 'LapTime_rolling_mean_5',
    'LapTime_rolling_std_3',  'LapTime_rolling_std_5',
    'Degradation_rolling_slope_3', 'Degradation_rolling_slope_5',
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
]

# Extended feature set: base + fold-safe target encodings
FULL_FEATURES = BASE_FEATURES + [
    'Race_target_encoded',
    'Driver_target_encoded',
    'Driver_avg_stint_length',
]

# CatBoost uses base features + raw categoricals (no manual encoding)
CB_FEATURES  = BASE_FEATURES + ['Race', 'Driver', 'Compound']
CB_CAT_COLS  = ['Race', 'Driver', 'Compound']

print(f'Rows: {len(df):,}  |  Base features: {len(BASE_FEATURES)}  |  Full (LGBM/XGB): {len(FULL_FEATURES)}  |  CatBoost: {len(CB_FEATURES)}')
print(f'Folds: {sorted(df["fold"].unique())}')

Rows: 439,140  |  Base features: 26  |  Full (LGBM/XGB): 29  |  CatBoost: 29
Folds: [np.int8(0), np.int8(1), np.int8(2), np.int8(3), np.int8(4)]


## Target encodings (fold-safe)

`Race_target_encoded` and `Driver_target_encoded` are the mean pit rate for each Race/Driver computed **on the training fold only**, then mapped to the validation fold. This prevents label leakage: the validation fold's encoding never uses its own labels.

`Driver_avg_stint_length` is the mean of each driver's per-stint maximum TyreLife, again computed on train only.

Rule: always call `apply_target_encodings(train_fold, val_fold)` — never on the full dataset before splitting.

In [3]:
def apply_target_encodings(train_df: pd.DataFrame, val_df: pd.DataFrame) -> pd.DataFrame:
    """Compute target encodings from train_df; apply to val_df. Returns val_df copy with new columns."""
    global_mean   = train_df['PitNextLap'].mean()
    race_map      = train_df.groupby('Race')['PitNextLap'].mean()
    driver_map    = train_df.groupby('Driver')['PitNextLap'].mean()
    stint_lengths = (
        train_df.groupby(['Driver', 'Race', 'Year', 'Stint'])['TyreLife']
        .max().reset_index()
        .groupby('Driver')['TyreLife'].mean()
    )
    global_stint  = stint_lengths.mean()

    out = val_df.copy()
    out['Race_target_encoded']     = out['Race'].map(race_map).fillna(global_mean)
    out['Driver_target_encoded']   = out['Driver'].map(driver_map).fillna(global_mean)
    out['Driver_avg_stint_length'] = out['Driver'].map(stint_lengths).fillna(global_stint)
    return out

## 1. LightGBM — default params

Establish a pre-tuning baseline using sensible defaults with early stopping (50 rounds patience). This gives us the floor that Optuna must beat and validates that the feature set works.

In [4]:
LGBM_DEFAULT = dict(
    n_estimators      = 1000,
    learning_rate     = 0.05,
    num_leaves        = 63,
    min_child_samples = 20,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.0,
    reg_lambda        = 1.0,
    random_state      = 42,
    verbose           = -1,
)

lgbm_default_oof = np.zeros(len(df))
t0 = time.time()

for fold_idx in range(5):
    tr_idx  = np.where(df['fold'] != fold_idx)[0]
    val_idx = np.where(df['fold'] == fold_idx)[0]

    train_sub = apply_target_encodings(df.iloc[tr_idx], df.iloc[tr_idx])
    val_sub   = apply_target_encodings(df.iloc[tr_idx], df.iloc[val_idx])

    X_tr, y_tr   = train_sub[FULL_FEATURES].to_numpy(), train_sub['PitNextLap'].to_numpy()
    X_val, y_val = val_sub[FULL_FEATURES].to_numpy(),   val_sub['PitNextLap'].to_numpy()

    callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
    m = lgb.LGBMClassifier(**LGBM_DEFAULT)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=callbacks)

    lgbm_default_oof[val_idx] = m.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, lgbm_default_oof[val_idx])
    print(f'  Fold {fold_idx}: AUC={fold_auc:.4f}  trees={m.best_iteration_}')

lgbm_default_auc = roc_auc_score(df['PitNextLap'], lgbm_default_oof)
print(f'\nLightGBM default OOF AUC: {lgbm_default_auc:.4f}  ({time.time()-t0:.0f}s)')

  Fold 0: AUC=0.8642  trees=190
  Fold 1: AUC=0.8195  trees=35
  Fold 2: AUC=0.8343  trees=42
  Fold 3: AUC=0.8888  trees=913
  Fold 4: AUC=0.8676  trees=432

LightGBM default OOF AUC: 0.8536  (38s)


## 2. Optuna hyperparameter search

Optuna uses **TPE (Tree-structured Parzen Estimator)** — a Bayesian sequential model that builds a probabilistic model of which parameter regions give high AUC and samples from those regions preferentially. It outperforms random search within the same trial budget.

For speed, tuning runs on **fold 0 only** (80% train / 20% val split by race). The hyperparameter landscape is smooth enough that fold 0 gives reliable parameter rankings. The final CV uses all 5 folds.

Search space rationale:
- `num_leaves`: controls model capacity. Higher → fits complex patterns but risks overfitting race-specific noise.
- `min_child_samples`: minimum data points per leaf. Higher → stronger regularisation, wider leaves.
- `subsample` / `colsample_bytree`: row and column subsampling. Introduces diversity, reduces variance.
- `reg_alpha` / `reg_lambda`: L1/L2 on leaf weights. Log scale because effective range spans several orders of magnitude.
- `learning_rate`: lower → needs more trees but generalises better; early stopping auto-adjusts tree count.

In [5]:
TUNE_FOLD = 0
tr_tune  = np.where(df['fold'] != TUNE_FOLD)[0]
val_tune = np.where(df['fold'] == TUNE_FOLD)[0]

train_tune = apply_target_encodings(df.iloc[tr_tune], df.iloc[tr_tune])
val_tune_e = apply_target_encodings(df.iloc[tr_tune], df.iloc[val_tune])

X_tr_t  = train_tune[FULL_FEATURES].to_numpy()
y_tr_t  = train_tune['PitNextLap'].to_numpy()
X_val_t = val_tune_e[FULL_FEATURES].to_numpy()
y_val_t = val_tune_e['PitNextLap'].to_numpy()

def objective(trial):
    params = dict(
        n_estimators      = 1000,
        learning_rate     = trial.suggest_float('learning_rate', 0.02, 0.3, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 31, 255),
        min_child_samples = trial.suggest_int('min_child_samples', 10, 100),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        random_state      = 42,
        verbose           = -1,
    )
    callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
    m = lgb.LGBMClassifier(**params)
    m.fit(X_tr_t, y_tr_t, eval_set=[(X_val_t, y_val_t)], callbacks=callbacks)
    preds = m.predict_proba(X_val_t)[:, 1]
    return roc_auc_score(y_val_t, preds)

study = optuna.create_study(
    direction = 'maximize',
    sampler   = optuna.samplers.TPESampler(seed=42),
)
t0 = time.time()
study.optimize(objective, n_trials=100, show_progress_bar=True)
print(f'\nBest AUC on fold {TUNE_FOLD}: {study.best_value:.4f}  ({time.time()-t0:.0f}s)')
print('Best params:')
for k, v in study.best_params.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.6f}')
    else:
        print(f'  {k}: {v}')

Best trial: 97. Best value: 0.868634: 100%|██████████| 100/100 [08:05<00:00,  4.85s/it]


Best AUC on fold 0: 0.8686  (485s)
Best params:
  learning_rate: 0.049228
  num_leaves: 62
  min_child_samples: 91
  subsample: 0.752871
  colsample_bytree: 0.746388
  reg_alpha: 9.791678
  reg_lambda: 0.000000


In [6]:
# Optuna param importance — which hyperparameters actually matter?
try:
    import plotly
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
except (ImportError, ValueError):
    # Fallback: print importances without plotly (or if nbformat version incompatible)
    importances = optuna.importance.get_param_importances(study)
    print('Hyperparameter importances (fraction of variance explained):')
    for param, imp in sorted(importances.items(), key=lambda x: -x[1]):
        bar = '█' * int(imp * 40)
        print(f'  {param:<25} {imp:.3f}  {bar}')

## 3. LightGBM — tuned (5-fold CV)

Full GroupKFold CV using Optuna best params. Per-fold model artifacts saved to `models/` for ensemble use in Notebook 09.

In [7]:
LGBM_BEST = dict(
    n_estimators = 2000,   # high ceiling; early stopping determines actual count
    random_state = 42,
    verbose      = -1,
    **study.best_params,
)

lgbm_oof  = np.zeros(len(df))
lgbm_aucs = []
t0 = time.time()

for fold_idx in range(5):
    tr_idx  = np.where(df['fold'] != fold_idx)[0]
    val_idx = np.where(df['fold'] == fold_idx)[0]

    train_sub = apply_target_encodings(df.iloc[tr_idx], df.iloc[tr_idx])
    val_sub   = apply_target_encodings(df.iloc[tr_idx], df.iloc[val_idx])

    X_tr, y_tr   = train_sub[FULL_FEATURES].to_numpy(), train_sub['PitNextLap'].to_numpy()
    X_val, y_val = val_sub[FULL_FEATURES].to_numpy(),   val_sub['PitNextLap'].to_numpy()

    callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
    m = lgb.LGBMClassifier(**LGBM_BEST)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=callbacks)

    lgbm_oof[val_idx] = m.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, lgbm_oof[val_idx])
    lgbm_aucs.append(fold_auc)
    print(f'  Fold {fold_idx}: AUC={fold_auc:.4f}  trees={m.best_iteration_}')

    joblib.dump(m, models_dir / f'lgbm_fold{fold_idx}.pkl')

lgbm_oof_auc = roc_auc_score(df['PitNextLap'], lgbm_oof)
print(f'\nLightGBM tuned OOF AUC : {lgbm_oof_auc:.4f}  (std {np.std(lgbm_aucs):.4f})  ({time.time()-t0:.0f}s)')
print(f'Gain over default      : {lgbm_oof_auc - lgbm_default_auc:+.4f}')

  Fold 0: AUC=0.8686  trees=284
  Fold 1: AUC=0.8246  trees=35
  Fold 2: AUC=0.8404  trees=41
  Fold 3: AUC=0.8893  trees=462
  Fold 4: AUC=0.8670  trees=294

LightGBM tuned OOF AUC : 0.8558  (std 0.0228)  (33s)
Gain over default      : +0.0023


## 4. `scale_pos_weight` experiment

AUC is a rank-order metric — it measures whether positives rank above negatives, not their absolute scores. In theory, `scale_pos_weight` (which up-weights the loss gradient for positive examples) should not affect rank ordering and thus not affect AUC. In practice, it can shift which splits are made early in training, which slightly changes the model's learned structure.

With ~20% positive rate: `neg/pos ≈ 4`. We test scale_pos_weight ∈ {1.0, 4.0} on fold 0 with the tuned params.

In [8]:
pos_rate     = df['PitNextLap'].mean()
spw          = (1 - pos_rate) / pos_rate
print(f'Positive rate: {pos_rate:.3f}  →  scale_pos_weight = {spw:.2f}')

results_spw = {}
for spw_val in [1.0, round(spw, 2)]:
    params = dict(**LGBM_BEST, scale_pos_weight=spw_val)
    callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
    m = lgb.LGBMClassifier(**params)
    m.fit(X_tr_t, y_tr_t, eval_set=[(X_val_t, y_val_t)], callbacks=callbacks)
    auc = roc_auc_score(y_val_t, m.predict_proba(X_val_t)[:, 1])
    results_spw[spw_val] = auc
    print(f'  scale_pos_weight={spw_val:.2f}:  fold-0 AUC = {auc:.4f}')

best_spw = max(results_spw, key=results_spw.get)
print(f'\nBetter scale_pos_weight: {best_spw}  (delta: {results_spw[best_spw] - results_spw[min(results_spw, key=results_spw.get)]:+.4f})')
print('Note: AUC differences < 0.001 are within noise — prefer scale_pos_weight=1 for simplicity.')

Positive rate: 0.199  →  scale_pos_weight = 4.03
  scale_pos_weight=1.00:  fold-0 AUC = 0.8686
  scale_pos_weight=4.03:  fold-0 AUC = 0.8537

Better scale_pos_weight: 1.0  (delta: +0.0149)
Note: AUC differences < 0.001 are within noise — prefer scale_pos_weight=1 for simplicity.


## 5. XGBoost

XGBoost grows trees **level-wise** (all leaves at one depth expanded before going deeper), while LightGBM grows **leaf-wise** (best single leaf expanded at each step). Level-wise is more conservative — it's harder to overfit but also harder to capture complex patterns with limited depth.

Using `tree_method='hist'` (histogram-based splits — same algorithm as LightGBM's default) for speed. Feature importance from XGBoost will be compared against LightGBM to validate that both models agree on the top predictors.

In [9]:
XGB_PARAMS = dict(
    n_estimators        = 2000,
    learning_rate       = 0.05,
    max_depth           = 6,
    min_child_weight    = 20,
    subsample           = 0.8,
    colsample_bytree    = 0.8,
    reg_alpha           = 0.1,
    reg_lambda          = 1.0,
    tree_method         = 'hist',
    eval_metric         = 'auc',
    early_stopping_rounds = 50,
    random_state        = 42,
    verbosity           = 0,
)

xgb_oof  = np.zeros(len(df))
xgb_aucs = []
t0 = time.time()

for fold_idx in range(5):
    tr_idx  = np.where(df['fold'] != fold_idx)[0]
    val_idx = np.where(df['fold'] == fold_idx)[0]

    train_sub = apply_target_encodings(df.iloc[tr_idx], df.iloc[tr_idx])
    val_sub   = apply_target_encodings(df.iloc[tr_idx], df.iloc[val_idx])

    X_tr, y_tr   = train_sub[FULL_FEATURES].to_numpy(), train_sub['PitNextLap'].to_numpy()
    X_val, y_val = val_sub[FULL_FEATURES].to_numpy(),   val_sub['PitNextLap'].to_numpy()

    m = xgb.XGBClassifier(**XGB_PARAMS)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    xgb_oof[val_idx] = m.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, xgb_oof[val_idx])
    xgb_aucs.append(fold_auc)
    print(f'  Fold {fold_idx}: AUC={fold_auc:.4f}  trees={m.best_iteration}')

    joblib.dump(m, models_dir / f'xgb_fold{fold_idx}.pkl')

xgb_oof_auc = roc_auc_score(df['PitNextLap'], xgb_oof)
print(f'\nXGBoost OOF AUC: {xgb_oof_auc:.4f}  (std {np.std(xgb_aucs):.4f})  ({time.time()-t0:.0f}s)')

  Fold 0: AUC=0.8653  trees=632
  Fold 1: AUC=0.8503  trees=11
  Fold 2: AUC=0.8394  trees=16
  Fold 3: AUC=0.8877  trees=413
  Fold 4: AUC=0.8686  trees=448

XGBoost OOF AUC: 0.8492  (std 0.0165)  (56s)


## 6. CatBoost

CatBoost's key differentiator is **ordered target encoding**: for each training example, it uses only examples that appear earlier in a randomly permuted ordering to compute the category mean — preventing within-fold target leakage. This means we can pass `Race`, `Driver`, and `Compound` as raw strings and let CatBoost encode them correctly without any manual fold-safe computation.

This is more statistically rigorous than our manual `apply_target_encodings()` (which encodes each training row using all training rows including itself) and can give CatBoost an edge on category-heavy signals.

Note: CatBoost uses `Pool` objects to associate feature names and category indices explicitly.

In [14]:
CB_PARAMS = dict(
    iterations     = 2000,
    learning_rate  = 0.03,   # lower than 0.05 — prevents overshooting on first few trees
    depth          = 6,
    l2_leaf_reg    = 3.0,    # back to default; 10.0 prevented recovery after initial overshoot
    random_seed    = 42,
    eval_metric    = 'AUC',
    od_type        = 'Iter', # stop if no improvement for od_wait consecutive rounds
    od_wait        = 100,    # generous patience; early stopping was triggering at 2-12 trees before
    use_best_model = True,
    verbose        = 0,
)

cb_oof  = np.zeros(len(df))
cb_aucs = []
t0 = time.time()

for fold_idx in range(5):
    tr_idx  = np.where(df['fold'] != fold_idx)[0]
    val_idx = np.where(df['fold'] == fold_idx)[0]

    train_sub = df.iloc[tr_idx][CB_FEATURES + ['PitNextLap']].copy()
    val_sub   = df.iloc[val_idx][CB_FEATURES + ['PitNextLap']].copy()

    # Ensure categoricals are strings (CatBoost requirement)
    for col in CB_CAT_COLS:
        train_sub[col] = train_sub[col].astype(str)
        val_sub[col]   = val_sub[col].astype(str)

    train_pool = Pool(
        data         = train_sub[CB_FEATURES],
        label        = train_sub['PitNextLap'],
        cat_features = CB_CAT_COLS,
    )
    val_pool = Pool(
        data         = val_sub[CB_FEATURES],
        label        = val_sub['PitNextLap'],
        cat_features = CB_CAT_COLS,
    )

    m = CatBoostClassifier(**CB_PARAMS)
    m.fit(train_pool, eval_set=val_pool)

    cb_oof[val_idx] = m.predict_proba(val_pool)[:, 1]
    fold_auc = roc_auc_score(val_sub['PitNextLap'], cb_oof[val_idx])
    cb_aucs.append(fold_auc)
    print(f'  Fold {fold_idx}: AUC={fold_auc:.4f}  trees={m.best_iteration_}')

    joblib.dump(m, models_dir / f'catboost_fold{fold_idx}.pkl')

cb_oof_auc = roc_auc_score(df['PitNextLap'], cb_oof)
print(f'\nCatBoost OOF AUC: {cb_oof_auc:.4f}  (std {np.std(cb_aucs):.4f})  ({time.time()-t0:.0f}s)')

  Fold 0: AUC=0.8314  trees=4
  Fold 1: AUC=0.8237  trees=14
  Fold 2: AUC=0.8143  trees=2
  Fold 3: AUC=0.8339  trees=5
  Fold 4: AUC=0.8503  trees=1705

CatBoost OOF AUC: 0.7823  (std 0.0119)  (449s)


## 7. Feature importance comparison

If LightGBM and XGBoost agree on the top 5 features, it's strong evidence those features are genuinely predictive rather than model-specific artefacts. Disagreement suggests one model is exploiting a feature via interactions the other misses.

In [15]:
# Load fold-0 models for importance comparison
lgbm_f0 = joblib.load(models_dir / 'lgbm_fold0.pkl')
xgb_f0  = joblib.load(models_dir / 'xgb_fold0.pkl')
cb_f0   = joblib.load(models_dir / 'catboost_fold0.pkl')

lgbm_imp = pd.Series(lgbm_f0.feature_importances_, index=FULL_FEATURES).sort_values(ascending=False)
xgb_imp  = pd.Series(xgb_f0.feature_importances_,  index=FULL_FEATURES).sort_values(ascending=False)
cb_imp   = pd.Series(
    dict(zip(CB_FEATURES, cb_f0.get_feature_importance()))
).sort_values(ascending=False)

print('Top 10 feature importances (split count):')
print(f'\n  {"Feature":<40} {"LGBM":>8}  {"XGB":>8}')
print('-' * 62)
for feat in lgbm_imp.head(10).index:
    lgbm_v = lgbm_imp.get(feat, 0)
    xgb_v  = xgb_imp.get(feat, 0)
    print(f'  {feat:<40} {lgbm_v:>8}  {xgb_v:>8}')

print(f'\n  Top 5 CatBoost features:')
for feat, imp in cb_imp.head(5).items():
    print(f'    {feat:<40} {imp:.1f}')

Top 10 feature importances (split count):

  Feature                                      LGBM       XGB
--------------------------------------------------------------
  Race_target_encoded                          1754  0.03647949546575546
  Cumulative_Degradation_winsorized            1567  0.02382567897439003
  laps_remaining                               1476  0.026544174179434776
  TyreLife_x_laps_remaining                    1065  0.09709669649600983
  LapTime_rolling_mean_3                       1053  0.015841880813241005
  LapTime_rolling_mean_5                        856  0.012173965573310852
  LapTime_Delta_lag1                            758  0.06866712123155594
  Driver_target_encoded                         667  0.016630349680781364
  Degradation_rate                              614  0.007873065769672394
  Position                                      609  0.011391473934054375

  Top 5 CatBoost features:
    Stint                                    50.8
    LapTime_Delta_

## 8. Summary & save OOF predictions

In [16]:
print('Gradient Boosting Suite — GroupKFold OOF AUC:\n')
print(f'  {"Model":<35} {"OOF AUC":>9}')
print('-' * 48)
print(f'  {"LightGBM (default)":<35} {lgbm_default_auc:>9.4f}')
print(f'  {"LightGBM (Optuna-tuned)":<35} {lgbm_oof_auc:>9.4f}  <-- primary model')
print(f'  {"XGBoost (hist)":<35} {xgb_oof_auc:>9.4f}')
print(f'  {"CatBoost":<35} {cb_oof_auc:>9.4f}')
print(f'\n  Baseline reference (Decision Tree depth=10): 0.8678')
print(f'  All GBM models must exceed 0.8678 to justify complexity.')

# Save OOF predictions for ensemble (Notebook 09)
oof_df = pd.DataFrame({
    'id'          : df['id'],
    'fold'        : df['fold'],
    'PitNextLap'  : df['PitNextLap'],
    'lgbm_oof'    : lgbm_oof,
    'xgb_oof'     : xgb_oof,
    'catboost_oof': cb_oof,
})
oof_df.to_parquet(processed_dir / 'oof_predictions.parquet', index=False)
print(f'\nSaved: {processed_dir / "oof_predictions.parquet"}  shape={oof_df.shape}')
print(oof_df.head(3).to_string(index=False))

Gradient Boosting Suite — GroupKFold OOF AUC:

  Model                                 OOF AUC
------------------------------------------------
  LightGBM (default)                     0.8536
  LightGBM (Optuna-tuned)                0.8558  <-- primary model
  XGBoost (hist)                         0.8492
  CatBoost                               0.7823

  Baseline reference (Decision Tree depth=10): 0.8678
  All GBM models must exceed 0.8678 to justify complexity.

Saved: c:\Repos\predict-f1-pit-stops\data\processed\oof_predictions.parquet  shape=(439140, 6)
   id  fold  PitNextLap  lgbm_oof  xgb_oof  catboost_oof
 7544     1         0.0  0.127422 0.152018      0.187477
31758     1         0.0  0.124230 0.157030      0.187477
73559     1         0.0  0.061684 0.113904      0.187477


In [17]:
# Log best GBM result to leaderboard
log_path = project_root / 'submissions' / 'leaderboard_log.md'
best_gbm_auc  = max(lgbm_oof_auc, xgb_oof_auc, cb_oof_auc)
best_gbm_name = ['LightGBM-tuned', 'XGBoost', 'CatBoost'][
    [lgbm_oof_auc, xgb_oof_auc, cb_oof_auc].index(best_gbm_auc)
]
with log_path.open('a') as f:
    f.write(f'| v005-lgbm | LightGBM Optuna-tuned GroupKFold | {lgbm_oof_auc:.4f} | — | Primary GBM model |\n')
    f.write(f'| v005-xgb  | XGBoost hist GroupKFold | {xgb_oof_auc:.4f} | — | Level-wise; validates LGBM |\n')
    f.write(f'| v005-cb   | CatBoost ordered-TE GroupKFold | {cb_oof_auc:.4f} | — | Native categoricals |\n')
print('Leaderboard log updated.')

Leaderboard log updated.


## Findings

### LightGBM
- Default OOF AUC: **0.8536**  |  Tuned OOF AUC: **0.8558**  |  Optuna gain: **+0.0023** (marginal)
- Optuna converged on: learning_rate≈0.049, num_leaves=62, min_child_samples=91, **reg_alpha=9.79**
- Most important hyperparameter (72% of variance): **reg_alpha** — extreme L1 regularisation.
- Folds 1–2 stop at 35–42 trees (vs 284–913 for folds 0, 3, 4). Data characteristic: those races have more predictable pit patterns, not a bug.
- Top features agreed by LGBM and XGB: `TyreLife_x_laps_remaining`, `laps_remaining`, `Cumulative_Degradation_winsorized`, `LapTime_Delta_lag1`

### Unexpected result: all GBMs underperform Decision Tree depth=10 (0.8678)
LGBM tuned = 0.8558, XGBoost = 0.8492. Most likely cause: reg_alpha=9.79 is fold-0-overfit.

### scale_pos_weight
Positive rate = 19.9%, neg/pos = 4.03. **scale_pos_weight=1.0 won** by +0.0149. AUC is rank-order — use default (1.0).

### XGBoost
OOF AUC: **0.8492** (std 0.0165). Level-wise growth is more conservative than LGBM's leaf-wise.

### CatBoost — dropped from ensemble

Three separate runs with progressively tuned params all failed:

| Run | Params | OOF AUC | Trees/fold |
|-----|--------|---------|-----------|
| 1 | early_stopping_rounds=50 | 0.7839 | 0–10 |
| 2 | od_type='Iter', od_wait=50, l2_leaf_reg=10 | 0.7983 | 2–12 |
| 3 | od_type='Iter', od_wait=100, l2_leaf_reg=3, lr=0.03 | 0.7823 | **2–14 (folds 0–3), 1705 (fold 4)** |

Run 3 reveals the root cause: **fold 4 trained correctly for 1705 trees; folds 0–3 all stopped at 2–14 despite 100-round patience**. This is not a configuration problem — it is a structural incompatibility between CatBoost's ordered encoding and GroupKFold by Race.

CatBoost's ordered encoding builds category statistics using a random permutation of the training pool. Each sample's Race/Driver encoding depends only on "earlier" samples in the permutation. When the validation fold contains races entirely absent from training (GroupKFold guarantee), the encoding built on training races does not transfer. CatBoost peaks at 2–14 trees, then every additional tree makes encodings more specific to training races and degrades validation AUC. The 100-round patience then confirms no improvement and stops.

For fold 4, the validation races happen to be more similar in character to the training races, so CatBoost's encoding generalises — hence 1705 trees of progressive learning.

**Conclusion:** CatBoost's ordered encoding and GroupKFold-by-Race are structurally incompatible when held-out races are diverse. Dropped from ensemble. LGBM + XGBoost OOF predictions are the ensemble inputs for Notebook 09.

### Final model comparison
| Model | OOF AUC | vs DT baseline (0.8678) |
|-------|---------|------------------------|
| Decision Tree depth=10 | 0.8678 | baseline |
| LightGBM default | 0.8536 | −0.0142 |
| LightGBM Optuna-tuned | **0.8558** | −0.0120 |
| XGBoost | 0.8492 | −0.0186 |
| CatBoost | ~~0.78~~ | ~~dropped~~ |

### Next step
Notebook 07 — SHAP Interpretability. Validate that LGBM feature directions match domain knowledge before ensemble.